In [45]:
import pandas as pd
import torch
import torch.nn as nn
import pickle

# =================================================================
# 1. RECRIAR A ESTRTUTRA
# =================================================================
class TextDNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(TextDNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.3)
        
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.2)
        
        self.fc3 = nn.Linear(64, num_classes)
        # Nota: O PyTorch aplica o Softmax automaticamente na CrossEntropyLoss!

    def forward(self, x):
        x = self.drop1(self.relu1(self.fc1(x)))
        x = self.drop2(self.relu2(self.fc2(x)))
        x = self.fc3(x)
        return x

In [47]:
# =================================================================
# 2. CARREGAR OS DADOS E OS ARTEFACTOS (TF-IDF E MODELO)
# =================================================================
print("A carregar o ficheiro subm2.csv...")

df_teste = pd.read_csv('subm2.csv', sep=';')
df_teste['Text'] = df_teste['Text'].fillna("").astype(str)
textos_teste = df_teste['Text'].tolist()

print("A carregar o TF-IDF treinado...")
with open('tfidf_pytorch.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

print("A carregar os pesos da DNN...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextDNN(input_dim=10000, num_classes=5).to(device)
# Carregar os pesos em segurança
model.load_state_dict(torch.load('modelo_dnn_pytorch.pth', map_location=device, weights_only=True))
model.eval() # Modo de inferência (desliga o Dropout)

A carregar o ficheiro subm2.csv...
A carregar o TF-IDF treinado...
A carregar os pesos da DNN...


TextDNN(
  (fc1): Linear(in_features=10000, out_features=128, bias=True)
  (relu1): ReLU()
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (relu2): ReLU()
  (drop2): Dropout(p=0.2, inplace=False)
  (fc3): Linear(in_features=64, out_features=5, bias=True)
)

In [48]:
# =================================================================
# 3. TRANSFORMAR OS TEXTOS E FAZER AS PREVISÕES
# =================================================================
print("\nA processar os textos e a gerar as previsões...")
# Transformar os textos em números usando o vocabulário do treino
X_tfidf_teste = vectorizer.transform(textos_teste).toarray()
X_tensor_teste = torch.tensor(X_tfidf_teste, dtype=torch.float32).to(device)

# Fazer a previsão matemática
with torch.no_grad():
    outputs = model(X_tensor_teste)
    preds_idx = outputs.argmax(dim=1).cpu().numpy()

# Traduzir os números para os nomes das IA's
inverse_label_map = {0: 'Human', 1: 'OpenAI', 2: 'Google', 3: 'Meta', 4: 'Anthropic'}
df_teste['Label'] = [inverse_label_map[idx] for idx in preds_idx]


A processar os textos e a gerar as previsões...


In [49]:
# =================================================================
# 4. EXPORTAR OS RESULTADOS E VER ESTATÍSTICAS
# =================================================================
nome_ficheiro = 'subm2-g7-MEI-A.csv'
df_teste[['ID', 'Label']].to_csv(nome_ficheiro, index=False, sep=';')

print("\nDistribuição FINAL das previsões no subm2:")
print(df_teste['Label'].value_counts())


Distribuição FINAL das previsões no subm2:
Label
Human        61
Anthropic    53
OpenAI       30
Google        6
Name: count, dtype: int64
